In [ ]:
%%writefile ./endpoint.yaml
$schema: https://azuremlschemas.azureedge.net/latest/managedOnlineEndpoint.schema.json
name: AllRecipeEndpoint
auth_mode: key

In [ ]:
!az ml online-endpoint create --file ./endpoint.yaml

In [ ]:
%%writefile ./deployment.yaml
$schema: https://azuremlschemas.azureedge.net/latest/managedOnlineDeployment.schema.json
name: blue
endpoint_name: AllRecipeEndpoint
model: azureml:phi3_finetuned_allrecipes@latest
code_configuration:
  code: .
  scoring_script: score_allrecipes.py
environment: azureml:phi35vision@latest
instance_type: Standard_NC12s_v3
instance_count: 1
request_settings:
  request_timeout_ms: 180000

In [ ]:
!az ml online-deployment create --all-traffic --file ./deployment.yaml

# Test the Deployment

In [12]:
import urllib.request
import json
import os
import ssl
from score_allrecipes import process_actions_string, image_to_data_url, load_image

# Replace this with the primary/secondary key, AMLToken, or Microsoft Entra ID token for the endpoint
api_key = 'dwgHkNprbQ8YJeq3YSN5mTrfoatcM2p4'
url = 'https://allrecipeendpoint.northeurope.inference.ml.azure.com/score'

def allowSelfSignedHttps(allowed):
    # bypass the server certificate verification on client side
    if allowed and not os.environ.get('PYTHONHTTPSVERIFY', '') and getattr(ssl, '_create_unverified_context', None):
        ssl._create_default_https_context = ssl._create_unverified_context

allowSelfSignedHttps(True) # this line is needed if you use self-signed certificate in your scoring service.

images_root_dir = "C:/Users/antonslutsky/Dev/azureml-quickstart/slm-finetuning/phi3-vision-finetune/applications/AllrecipesAgent/output/session_2024-08-31_11-24-25/images"
images_ext = ".png"

def local_image_loader(local_image_name):
    return load_image(f"{images_root_dir}/{local_image_name}{images_ext}")

def local_image_action_updater(image, img_cnt, actions_image_url):
    image = local_image_loader(actions_image_url)
    return image_to_data_url(image, images_ext)

def test_slm(prompt = "You are a useful AI that searches AllRecipes.com website for various recipies.  The following json document contains a set of keyboard and mouth actions together with the screenshots that preempted them to search for 'italian wedding soup' recipe on the website.  Suggest the nest set of keyboard and mouth actions to continue searching for the recipe. ['<sleep>11.645689', '<image>screenshot_2024-08-31_11-24-34.961750', '<mouse>on_click(1070,182,Button.left,True)', '<sleep>3.881456', '<image>screenshot_2024-08-31_11-24-40.230237', 'italian', '<key>Key.space:True', 'wedding', '<key>Key.space:True']"):

    print("111111111111111")
    action_string, images = process_actions_string(prompt, 
                                                   image_loader=local_image_loader,
                                                   action_updater=local_image_action_updater)
    print("222222222222222")
    with open("test_out.txt", "w") as test_out:
        test_out.write(action_string)

    print("333333333333")
    data = {"input_data": {"input_string": [
            action_string
    ]}}
    print("4444444444444")
    body = str.encode(json.dumps(data))
    print("5555555555555")
    # print("body:", body)

    if not api_key:
        raise Exception("A key should be provided to invoke the endpoint")

    headers = {'Content-Type':'application/json', 'Authorization':('Bearer '+ api_key), 'azureml-model-deployment': 'blue' }

    if True:
        req = urllib.request.Request(url, body, headers)

        try:
            print("------------------------------------")
            response = urllib.request.urlopen(req)

            
            result = response.read()
            print(result)
            print("=====================================")
            #return result
        except urllib.error.HTTPError as error:
            print("The request failed with status code: " + str(error.code))

            # Print the headers - they include the requert ID and the timestamp, which are useful for debugging the failure
            print(error.info())
            print(error.read().decode("utf8", 'ignore'))


test_slm()

111111111111111
222222222222222
333333333333
4444444444444
5555555555555
------------------------------------
The request failed with status code: 424
server: azureml-frontdoor
date: Sun, 01 Sep 2024 17:18:17 GMT
content-type: application/json
content-length: 92
x-ms-run-function-failed: True
x-ms-server-version: azmlinfsrv/0.8.4.1
x-ms-request-id: e47be5e6-7c05-4547-831a-f12c107ccfd0
x-request-id: e47be5e6-7c05-4547-831a-f12c107ccfd0
ms-azureml-model-error-reason: model_error
ms-azureml-model-error-statuscode: 500
azureml-model-deployment: blue
connection: close


{"message": "An unexpected error occurred in scoring script. Check the logs for more info."}
